In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/spam.csv", encoding="latin-1")

df.columns = ["label", "message"]

df["label"] = df["label"].map({"spam": 1, "ham": 0})

df = df.dropna(subset=["label", "message"])
df["message"] = df["message"].astype(str)
df["label"] = df["label"].astype(int)

def pre_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'\d+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text
    else:
        return ""

df["message"] = df["message"].apply(pre_text)

X_train, X_test, y_train, y_test = train_test_split(df["message"], df["label"], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = DecisionTreeClassifier(max_depth=10, criterion="entropy", random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Training Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training Accuracy: 0.9354
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.99      0.96       965
           1       0.93      0.56      0.70       150

    accuracy                           0.94      1115
   macro avg       0.93      0.78      0.83      1115
weighted avg       0.94      0.94      0.93      1115



In [ ]:
new_df = pd.read_csv("/content/drive/MyDrive/unclassified.csv", encoding="latin-1")

new_df["message"] = new_df["message"].fillna("").astype(str)
new_df["message"] = new_df["message"].apply(pre_text)

new_df["Predicted Label"] = model.predict(vectorizer.transform(new_df["message"]))

new_df["Predicted Label"] = new_df["Predicted Label"].map({1: "Spam", 0: "Not Spam"})

new_df.to_csv("classified_emails.csv", index=False)

print("New email classifications saved to 'classified_emails.csv'.")

New email classifications saved to 'classified_emails.csv'.
